In [7]:
import time
import os
import requests
import pandas as pd
from tqdm import tqdm
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import (
    NoSuchElementException,
    TimeoutException,
    StaleElementReferenceException
)
from webdriver_manager.chrome import ChromeDriverManager

# Categories

In [8]:
CATEGORIES = {
    "women_hoodies_sweatshirts": "https://www.jumia.com.eg/mlp-women-hoodies-sweatshirts/",
    "women_jeans": "https://www.jumia.com.eg/womens-jeans/",
    "women_shoes": "https://www.jumia.com.eg/womens-shoes/",
    "women_bags": "https://www.jumia.com.eg/womens-bags/",
    "women_dresses": "https://www.jumia.com.eg/womens-dresses/",
    "women_skirts": "https://www.jumia.com.eg/womens-skirts/",
    "Women_Coats_Jackets_Vests": "https://www.jumia.com.eg/womens-coats-jackets-vests/",
    "women_sunglasses": "https://www.jumia.com.eg/womens-sunglasses/",
    "women_watches": "https://www.jumia.com.eg/womens-watches/",
    "women_jewelry": "https://www.jumia.com.eg/womens-jewelry/",
    "women_belts": "https://www.jumia.com.eg/womens-belts/",
    "women_wallets": "https://www.jumia.com.eg/womens-wallets/",
    "women_sandals": "https://www.jumia.com.eg/womens-sandals/",
    "women_boots": "https://www.jumia.com.eg/womens-boots/",
    "women_sneakers": "https://www.jumia.com.eg/womens-sneakers/",

    "Men_hoodies_Sweatshirts": "https://www.jumia.com.eg/mens-hoodies-sweatshirts/",
    "men_jeans": "https://www.jumia.com.eg/mens-jeans/",
    "men_shoes": "https://www.jumia.com.eg/mens-shoes/",
    "men_tshirts": "https://www.jumia.com.eg/mens-t-shirts/",
    "men_shirts": "https://www.jumia.com.eg/mens-shirts/",
    "Men_Jackets_Coats": "https://www.jumia.com.eg/mens-jackets-coats/",
    "men_suits": "https://www.jumia.com.eg/mens-suits/",
    "men_shorts": "https://www.jumia.com.eg/mens-shorts/",
    "men_watches": "https://www.jumia.com.eg/mens-watches/",
    "men_sunglasses": "https://www.jumia.com.eg/mens-sunglasses/",
    "men_belts": "https://www.jumia.com.eg/mens-belts/",
    "men_wallets": "https://www.jumia.com.eg/mens-wallets/",
    "men_sneakers": "https://www.jumia.com.eg/mens-sneakers/",
    "Men_half_boots": "https://www.jumia.com.eg/mens-boots/",
    "men_fashion_accessories": "https://www.jumia.com.eg/mens-fashion-accessories/",

    "kids_Boys_Fashion_Hoodies_Sweatshirts": "https://www.jumia.com.eg/boy-hoodies-sweatshirts/#catalog-listing",
    "kids_Boys_Fashion_Jeans_Pants": "https://www.jumia.com.eg/mlp-jeans-pants/",

    "kids_Girls_Fashion_Hoodies_Sweatshirts": "https://www.jumia.com.eg/girls-fashion-hoodies-sweatshirts/#catalog-listing",
    "kids_Girls_Fashion_Jeans_Pants": "https://www.jumia.com.eg/mlp-girls-pants-jeans/"

}

# Config

In [9]:
MAX_PAGES = 5
DELAY = 2

OUTPUT_DIR = "data"
IMAGE_DIR = os.path.join(OUTPUT_DIR, "images")
RAW_DIR = os.path.join(OUTPUT_DIR, "raw")
CSV_PATH = os.path.join(RAW_DIR, "products.csv")

# Driver

In [10]:
def create_driver():
    options = Options()
    options.add_argument('--headless=new')
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument('--window-size=1920,1080')
    # FIX 1: User-Agent so Jumia does not block headless Chrome
    options.add_argument(
        'user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
        'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    )
    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=options)
    return driver

# Scrape Page


In [11]:
def scrape_page(driver, url, category, gender):
    products = []
    try:
        driver.get(url)
        WebDriverWait(driver, 15).until(
            EC.presence_of_all_elements_located((By.CSS_SELECTOR, 'article.prd'))
        )
        time.sleep(2)  # let lazy-load images settle
        cards = driver.find_elements(By.CSS_SELECTOR, 'article.prd')

        for card in cards:
            try:
                title = card.find_element(By.CSS_SELECTOR, 'h3.name').text.strip()
                price = card.find_element(By.CSS_SELECTOR, 'div.prc').text.strip()

                img = card.find_element(By.CSS_SELECTOR, 'img.img')
                # FIX 2: Jumia lazy-loads — use data-src first, fall back to src
                img_url = img.get_attribute('data-src') or img.get_attribute('src') or ''

                product_url = card.find_element(By.CSS_SELECTOR, 'a.core').get_attribute('href')

                if not title or not img_url:
                    continue

                products.append({
                    'title': title,
                    'price': price,
                    'category': category,
                    'gender': gender,
                    'img_url': img_url,
                    'product_url': product_url,
                })
            except StaleElementReferenceException:
                continue
            except Exception:
                continue

    except TimeoutException:
        print(f'  Timeout: {url}')
    except Exception as e:
        print(f'  Error on {url}: {e}')

    return products

# Scrape All

In [12]:
def scrape_all_categories():
    os.makedirs(RAW_DIR, exist_ok=True)
    os.makedirs(IMAGE_DIR, exist_ok=True)
    driver = create_driver()
    all_products = []

    for cat_key, base_url in CATEGORIES.items():
        gender, category = cat_key.split('_', 1)
        print(f'Scraping: {cat_key}')

        for page in range(1, MAX_PAGES + 1):
            # FIX 3: correct pagination URL with #catalog-listing
            if page == 1:
                url = base_url
            else:
                base_clean = base_url.split('#')[0].rstrip('/') + '/'
                url = f'{base_clean}?page={page}#catalog-listing'

            print(f'  Page {page}', end=' ', flush=True)
            data = scrape_page(driver, url, category, gender)
            print(f'-> {len(data)} products')
            all_products.extend(data)
            time.sleep(2)

    driver.quit()
    print(f'Total scraped: {len(all_products)} products')
    return all_products

products = scrape_all_categories()

Scraping: women_hoodies_sweatshirts
  Page 1 -> 4 products
  Page 2 -> 40 products
  Page 3 -> 40 products
  Page 4 -> 40 products
  Page 5 -> 40 products
Scraping: women_jeans
  Page 1 -> 47 products
  Page 2 -> 47 products
  Page 3 -> 47 products
  Page 4 -> 47 products
  Page 5 -> 47 products
Scraping: women_shoes
  Page 1 -> 52 products
  Page 2 -> 52 products
  Page 3 -> 52 products
  Page 4 -> 52 products
  Page 5 -> 52 products
Scraping: women_bags
  Page 1 -> 52 products
  Page 2 -> 52 products
  Page 3 -> 52 products
  Page 4 -> 52 products
  Page 5 -> 52 products
Scraping: women_dresses
  Page 1 -> 47 products
  Page 2 -> 47 products
  Page 3 -> 47 products
  Page 4 -> 47 products
  Page 5 -> 47 products
Scraping: women_skirts
  Page 1 -> 41 products
  Page 2 -> 41 products
  Page 3 -> 41 products
  Page 4 -> 41 products
  Page 5 -> 41 products
Scraping: Women_Coats_Jackets_Vests
  Page 1 -> 45 products
  Page 2 -> 45 products
  Page 3 -> 45 products
  Page 4 -> 45 products
 

# Save CSV

In [13]:
def save_to_csv(products):
    df = pd.DataFrame(products)
    df.drop_duplicates(subset=["product_url"], inplace=True)
    df.to_csv(CSV_PATH, index=False)
    return df
df = save_to_csv(products)

In [15]:

df.head()

,title,price,category,gender,img_url,product_url
0,Decathlon Women's Relaxation Yoga Fleece Sweat...,"EGP 1,049.00",hoodies_sweatshirts,women,https://eg.jumia.is/unsafe/fit-in/300x300/filt...,https://www.jumia.com.eg/decathlon-womens-rela...
1,Decathlon Women's Fitness Sweatshirt 100 - Pink,EGP 699.00,hoodies_sweatshirts,women,https://eg.jumia.is/unsafe/fit-in/300x300/filt...,https://www.jumia.com.eg/decathlon-womens-fitn...
2,Decathlon Women's Fitness Hoodie 520 - Pink Qu...,"EGP 1,739.00",hoodies_sweatshirts,women,https://eg.jumia.is/unsafe/fit-in/300x300/filt...,https://www.jumia.com.eg/decathlon-womens-fitn...
3,Decathlon Women's Cropped Fitness Sweatshirt 5...,"EGP 1,159.00",hoodies_sweatshirts,women,https://eg.jumia.is/unsafe/fit-in/300x300/filt...,https://www.jumia.com.eg/decathlon-womens-crop...
4,LC Waikiki Crew Neck Daisy Duck Printed Long S...,EGP 329.00,hoodies_sweatshirts,women,https://eg.jumia.is/unsafe/fit-in/300x300/filt...,https://www.jumia.com.eg/lc-waikiki-crew-neck-...


# Download Images

In [14]:
def download_images(df):
    os.makedirs(IMAGE_DIR, exist_ok=True)
    headers = {'User-Agent': 'Mozilla/5.0'}

    for i, row in tqdm(df.iterrows(), total=len(df), desc='Downloading images'):
        folder = os.path.join(IMAGE_DIR, f"{row['gender']}_{row['category']}")
        os.makedirs(folder, exist_ok=True)
        path = os.path.join(folder, f'{i}.jpg')
        if os.path.exists(path):
            continue
        try:
            img = requests.get(row['img_url'], headers=headers, timeout=10)
            open(path, 'wb').write(img.content)
        except Exception:
            pass

download_images(df)